# 02 · Probability, Bayes & Distribution Fitting

**Business question.** Does the toss confer a material advantage? What is the right *probabilistic* model for a first-innings total and for wickets per match — and how confidently can we use those models for management-style probability statements?

**Course topics covered:** marginal/conditional probability, Bayes' theorem, Normal distribution, Poisson distribution, Shapiro–Wilk test, Kolmogorov–Smirnov test, chi-square goodness-of-fit.

In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_processed_or_build
from src.viz import savefig, PRIMARY, ACCENT, HIGHLIGHT, PALETTE

matches, deliveries, innings = load_processed_or_build()

## 2.1 Does the toss winner win the match more often?

In [2]:
m = matches.dropna(subset=['winner']).copy()
m['toss_winner_won'] = (m['toss_winner'] == m['winner']).astype(int)

p_overall = m['toss_winner_won'].mean()
n = len(m)
print(f'P(toss winner wins match) = {p_overall:.4f}  (n = {n})')

# 95% Wald CI for the proportion
se = np.sqrt(p_overall * (1 - p_overall) / n)
ci_lo, ci_hi = p_overall - 1.96*se, p_overall + 1.96*se
print(f'95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]')

P(toss winner wins match) = 0.5118  (n = 1184)
95% CI: [0.4834, 0.5403]


## 2.2 Conditional on the toss *decision*

In [3]:
by_decision = m.groupby('toss_decision')['toss_winner_won'].agg(['mean', 'count'])
by_decision.columns = ['P(win | chose this)', 'n']
by_decision

,P(win | chose this),n
toss_decision,,
bat,0.456576,403
field,0.540333,781


In [4]:
# Plot win-rate by toss decision
fig, ax = plt.subplots(figsize=(8, 4.5))
vals = by_decision['P(win | chose this)'].sort_index()
bars = ax.bar(vals.index, vals.values, color=[ACCENT, HIGHLIGHT])
for b, v in zip(bars, vals.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01,
            f'{v:.1%}', ha='center', fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', label='50% baseline')
ax.set_ylim(0, 0.75)
ax.set_title('Toss-winner Win-Rate by Toss Decision')
ax.set_ylabel('P(toss winner wins match)')
ax.legend()
savefig('prob_01_toss_decision.png')
plt.show()

## 2.3 Bayes' theorem — reverse the conditional

Question: given that the toss winner won the match, what is the probability they had chosen to *field*?

Bayes: $P(F | W) = \dfrac{P(W | F)\,P(F)}{P(W)}$

In [5]:
p_field = (m['toss_decision'] == 'field').mean()
p_win_given_field = m.loc[m['toss_decision'] == 'field', 'toss_winner_won'].mean()
p_win = m['toss_winner_won'].mean()
p_field_given_win = p_win_given_field * p_field / p_win

print(f'P(chose Field)                  = {p_field:.4f}')
print(f'P(W | chose Field)              = {p_win_given_field:.4f}')
print(f'P(W) overall                    = {p_win:.4f}')
print(f'=> P(chose Field | toss winner won) = {p_field_given_win:.4f}')

P(chose Field)                  = 0.6596
P(W | chose Field)              = 0.5403
P(W) overall                    = 0.5118
=> P(chose Field | toss winner won) = 0.6964


## 2.4 Fit a Normal distribution to first-innings totals

In [6]:
first_inn = innings[innings['inning'] == 1]
mu, sigma = stats.norm.fit(first_inn['innings_total'])
print(f'Fitted N(mu = {mu:.2f}, sigma = {sigma:.2f})')

# Shapiro-Wilk (caps at n=5000 in scipy)
sample = first_inn['innings_total'].sample(n=min(4000, len(first_inn)), random_state=42)
sw_W, sw_p = stats.shapiro(sample)
ks_D, ks_p = stats.kstest(first_inn['innings_total'], 'norm', args=(mu, sigma))
print(f'Shapiro-Wilk : W = {sw_W:.4f}, p = {sw_p:.4f}')
print(f'KS vs fitted : D = {ks_D:.4f}, p = {ks_p:.4f}')

Fitted N(mu = 167.36, sigma = 33.34)
Shapiro-Wilk : W = 0.9957, p = 0.0018
KS vs fitted : D = 0.0266, p = 0.3606


In [7]:
# Empirical vs theoretical
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.histplot(first_inn['innings_total'], bins=30, stat='density',
             color=PRIMARY, alpha=0.6, ax=ax, label='Empirical')
xs = np.linspace(first_inn['innings_total'].min(), first_inn['innings_total'].max(), 200)
ax.plot(xs, stats.norm.pdf(xs, mu, sigma), color=ACCENT, lw=2.5,
        label=f'N({mu:.0f}, {sigma:.0f})')
ax.set_title('First-Innings Totals: Empirical vs Fitted Normal')
ax.set_xlabel('Runs'); ax.set_ylabel('Density')
ax.legend()
savefig('prob_02_normal_fit.png')
plt.show()

## 2.5 Probability statements from the fitted Normal

In [8]:
thresholds = [140, 160, 180, 200, 220]
probs = pd.DataFrame({
    'Threshold (runs)': thresholds,
    'P(score < threshold)': [stats.norm.cdf(t, mu, sigma) for t in thresholds],
    'P(score > threshold)': [1 - stats.norm.cdf(t, mu, sigma) for t in thresholds],
}).round(4)
probs

,Threshold (runs),P(score < threshold),P(score > threshold)
0,140,0.2059,0.7941
1,160,0.4126,0.5874
2,180,0.6477,0.3523
3,200,0.8362,0.1638
4,220,0.9428,0.0572


## 2.6 Poisson fit to wickets per match

In [9]:
wkts = (deliveries.dropna(subset=['player_dismissed'])
        .groupby('match_id').size().rename('wickets_in_match').reset_index())
lam = wkts['wickets_in_match'].mean()
print(f'lambda (avg wickets per match) = {lam:.2f}')

obs = wkts['wickets_in_match'].value_counts().sort_index()
k = obs.index.to_numpy()
exp = stats.poisson.pmf(k, lam) * len(wkts)
exp = exp * (obs.sum() / exp.sum())
chi2, chi_p = stats.chisquare(f_obs=obs.values, f_exp=exp)
print(f'Chi-square GoF: chi2 = {chi2:.2f}, p = {chi_p:.4f}')

lambda (avg wickets per match) = 11.82
Chi-square GoF: chi2 = 36.14, p = 0.0148


In [10]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(k, obs.values, alpha=0.6, color=PRIMARY, label='Observed')
ax.plot(k, exp, color=ACCENT, marker='o', linewidth=2,
        label=f'Poisson(lambda = {lam:.2f}) expected')
ax.set_title('Wickets per Match: Observed vs Poisson')
ax.set_xlabel('Wickets in match'); ax.set_ylabel('Number of matches')
ax.legend()
savefig('prob_03_poisson_wickets.png')
plt.show()

## 2.7 Take-aways

* The toss confers a small, measurable edge — within the 95% CI we can say whether it is robustly above 50%.
* First-innings totals are well-approximated by a Normal distribution; the goodness-of-fit p-values tell us whether to trust that approximation for probability statements.
* The Poisson is a reasonable starting model for wickets per match. Departures (over-dispersion) would suggest a Negative Binomial as a richer next step.